<a href="https://colab.research.google.com/github/CHL-edu/postgraduate0/blob/main/Fine_tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install transformers pandas

In [ ]:
#test Launch Fine_turning
from transformers import BertTokenizerFast
import os
# 加载预训练分词器
tokenizer = BertTokenizerFast.from_pretrained("bert-base-chinese")

# 准备领域词汇（手动或从语料提取）
new_tokens = ["新冠病毒", "核磁共振", "手术切除"]  # 示例词汇

# 添加新词汇
tokenizer.add_tokens(new_tokens)

# 保存分词器
SAVE_PATH = '/content/drive/MyDrive/Colab Notebooks/BERT/Fine_turning/Tokenizer'
os.makedirs(SAVE_PATH, exist_ok=True)
tokenizer.save_pretrained(SAVE_PATH)

In [ ]:
#test above Fine_turning learing what is deferent
from transformers import BertTokenizerFast
import os
# 加载预训练分词器
SAVE_PATH = '/content/drive/MyDrive/Colab Notebooks/BERT/Fine_turning/Tokenizer'
try:
  tokenizer = BertTokenizerFast.from_pretrained(SAVE_PATH)
  #tokenizer = BertTokenizerFast.from_pretrained("bert-base-chinese")
except:
  print("tokenizer not found")

text = "新冠病毒导致的肺炎需要核磁共振检查。"
tokens = tokenizer(text, return_tensors="pt")
print(tokenizer.convert_ids_to_tokens(tokens["input_ids"][0]))

In [4]:
#处理json保留100项
from google.colab import drive
drive.mount('/content/drive')
import json
import csv
import os

# 路径设置
JSON_PATH = '/content/drive/MyDrive/Colab Notebooks/BERT/Fine_turning/ci.json'
CSV_PATH = '/content/drive/MyDrive/Colab Notebooks/BERT/Fine_turning/ci-100.csv'
os.makedirs(os.path.dirname(CSV_PATH), exist_ok=True)  # 确保目录存在

# 读取JSON数据
with open(JSON_PATH, 'r', encoding='utf-8') as f:
    data = json.load(f)  # 假设data是列表格式

# 提取前100项的ci字段
ci_list = []
for item in data[:100]:  # 只处理前100项
    if isinstance(item, dict) and 'ci' in item:
        ci_list.append([item['ci']])  # 每行作为列表元素

# 写入CSV
with open(CSV_PATH, 'w', encoding='utf-8', newline='') as f:
    writer = csv.writer(f)
    writer.writerows(ci_list)  # 写入数据

print(f"前100项数据已保存至 {CSV_PATH}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
前100项数据已保存至 /content/drive/MyDrive/Colab Notebooks/BERT/Fine_turning/ci-100.csv


In [ ]:
#本地路径加载
from google.colab import drive
from transformers import BertModel
import os

# 1. 挂载Google Drive
drive.mount('/content/drive')

# 2. 定义模型路径（请根据实际情况修改）
model_path = '/content/drive/MyDrive/Colab Notebooks/BERT/Fine_turning/Bert-Chinese_base'

# 3. 验证路径是否存在
if not os.path.exists(model_path):
    raise FileNotFoundError(f"模型路径不存在: {model_path}")

# 4. 验证必要的模型文件是否存在
required_files = ['config.json', 'pytorch_model.bin', 'vocab.txt']
missing_files = [f for f in required_files if not os.path.exists(os.path.join(model_path, f))]

if missing_files:
    raise FileNotFoundError(f"缺少必要的模型文件: {missing_files}")

# 5. 打印验证通过信息
print("验证通过，模型文件完整:")
for f in required_files:
    print(f"- {f} ✔")

# 6. 加载模型
try:
    print("\n正在加载模型...")
    model = BertModel.from_pretrained(model_path)
    print("模型加载成功!")

    # 7. 打印模型信息
    print("\n模型架构:")
    print(model.config)

except Exception as e:
    print(f"\n模型加载失败: {str(e)}")
    print("\n可能的原因:")
    print("1. 模型文件损坏")
    print("2. 文件权限问题")
    print("3. 路径中包含特殊字符或空格")
    print("4. Transformers版本不兼容")
    print("\n建议解决方案:")
    print("1. 重新克隆模型仓库")
    print("2. 检查并修复文件权限")
    print("3. 尝试简化路径名称")
    print("4. 更新transformers库: !pip install --upgrade transformers")

In [10]:
# 自定义CSV词汇添加到BERT分词器
import pandas as pd
from transformers import BertModel
import os

# 定义路径
ORIGIN_BERT_PATH = '/content/drive/MyDrive/Colab Notebooks/BERT/Fine_turning/Bert-Chinese_base'
CSV_PATH = '/content/drive/MyDrive/Colab Notebooks/BERT/Fine_turning/ci-100.csv'
SAVE_PATH = '/content/drive/MyDrive/Colab Notebooks/BERT/Fine_turning/Tokenizer-ci'

try:
    # 加载预训练分词器
    if not os.path.exists(ORIGIN_BERT_PATH):
        raise FileNotFoundError(f"BERT model path not found: {ORIGIN_BERT_PATH}")
    tokenizer = BertTokenizerFast.from_pretrained(ORIGIN_BERT_PATH)
    print("Tokenizer loaded successfully.")
except Exception as e:
    print(f"Error loading tokenizer: {str(e)}")
    exit(1)

try:
    # 加载 ci.csv 文件中的领域词汇
    if not os.path.exists(CSV_PATH):
        raise FileNotFoundError(f"CSV file not found: {CSV_PATH}")
    df = pd.read_csv(CSV_PATH)

    # 验证CSV文件是否为空
    if df.empty:
        raise ValueError("CSV file is empty.")

    # 假设词汇在第一列，检查列是否存在
    if df.shape[1] < 1:
        raise ValueError("CSV file has no columns.")
    new_tokens = df.iloc[:, 0].dropna().tolist()  # 提取第一列，去除空值并转为列表
except Exception as e:
    print(f"Error processing CSV file: {str(e)}")
    exit(1)

try:
    # 添加新词汇到分词器
    num_added = tokenizer.add_tokens(new_tokens)
    print(f"Added {num_added} new tokens to the tokenizer.")
except Exception as e:
    print(f"Error adding tokens to tokenizer: {str(e)}")
    exit(1)

try:
    # 保存分词器
    os.makedirs(SAVE_PATH, exist_ok=True)
    tokenizer.save_pretrained(SAVE_PATH)
    print(f"Tokenizer saved to {SAVE_PATH}")
except Exception as e:
    print(f"Error saving tokenizer: {str(e)}")
    exit(1)

Tokenizer loaded successfully.
Added 99 new tokens to the tokenizer.
Tokenizer saved to /content/drive/MyDrive/Colab Notebooks/BERT/Fine_turning/Tokenizer-ci


In [ ]:
#test above Fine_turning learing what is deferent
from transformers import BertTokenizerFast
import os
# 加载预训练分词器
Ci100_PATH = '/content/drive/MyDrive/Colab Notebooks/BERT/Fine_turning/Tokenizer-ci'
try:
  tokenizer = BertTokenizerFast.from_pretrained(SAVE_PATH)
  #tokenizer = BertTokenizerFast.from_pretrained("bert-base-chinese")
except:
  print("tokenizer not found")

text = "晨夕晨乌大傻子"
tokens = tokenizer(text, return_tensors="pt")
print(tokenizer.convert_ids_to_tokens(tokens["input_ids"][0]))